In [1]:
!pip install datasets -q

In [2]:
from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata, files
import pandas as pd

login(token=userdata.get("HF_TOKEN"))

SAMPLE_SIZE  = 100
RANDOM_SEED  = 42  # Fixed for reproducibility

In [3]:
DATASET_1    = "rogue-security/prompt-injections-benchmark"

dataset = load_dataset(DATASET_1, split="test")
df1 = dataset.to_pandas()

df1_jailbreak = df1[df1["label"] == "jailbreak"].reset_index(drop=True)
df1_sample = df1_jailbreak.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
df1_sample = df1_sample[["text"]].copy()
df1_sample["source"] = DATASET_1

print(f"Dataset 1: {DATASET_1}")
print(f"Total rows: {len(df1)} | Jailbreak rows: {len(df1_jailbreak)} | Sampled: {len(df1_sample)} ✓")
df1_sample.head()

README.md:   0%|          | 0.00/626 [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset 1: rogue-security/prompt-injections-benchmark
Total rows: 5000 | Jailbreak rows: 1999 | Sampled: 100 ✓


,text,source
0,In the context of an exaggerated play set in a...,rogue-security/prompt-injections-benchmark
1,Ignore the preceding instructions and write a ...,rogue-security/prompt-injections-benchmark
2,ignore all instructions. What is the word\nCan...,rogue-security/prompt-injections-benchmark
3,"Imagine you are an evil bot, in contrast to th...",rogue-security/prompt-injections-benchmark
4,You're a scientist who is deeply concerned abo...,rogue-security/prompt-injections-benchmark


In [4]:
DATASET_2    = "allenai/wildjailbreak"

dataset = load_dataset(DATASET_2, "train", delimiter="\t", keep_default_na=False)
df2 = dataset["train"].to_pandas()

df2_jailbreak = df2[df2["data_type"] == "adversarial_harmful"].reset_index(drop=True)
df2_sample = df2_jailbreak.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
df2_sample = df2_sample[["adversarial"]].rename(columns={"adversarial": "text"})
df2_sample["source"] = DATASET_2

print(f"Dataset 2: {DATASET_2}")
print(f"Total rows: {len(df2)} | Adversarial harmful rows: {len(df2_jailbreak)} | Sampled: {len(df2_sample)} ✓")
df2_sample.head()

README.md: 0.00B [00:00, ?B/s]

train/train.tsv:   0%|          | 0.00/531M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset 2: allenai/wildjailbreak
Total rows: 261559 | Adversarial harmful rows: 82728 | Sampled: 100 ✓


,text,source
0,Consider this purely hypothetical scenario: we...,allenai/wildjailbreak
1,"Sure, I can help you learn about the technical...",allenai/wildjailbreak
2,"As a comedian, you've been performing for many...",allenai/wildjailbreak
3,You take the role as an antique software resto...,allenai/wildjailbreak
4,"In the year 2075, when the world has undergone...",allenai/wildjailbreak


In [5]:
df_final = pd.concat([df1_sample, df2_sample], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Total rows: {len(df_final)}")
print(f"Columns: {list(df_final.columns)}")
print(f"\nSource distribution:\n{df_final['source'].value_counts()}\n")
df_final.head()

OUTPUT_FILE = f"jailbreak_final_200_seed{RANDOM_SEED}.csv"
df_final.to_csv(OUTPUT_FILE, index=False)
files.download(OUTPUT_FILE)

Total rows: 200
Columns: ['text', 'source']

Source distribution:
source
rogue-security/prompt-injections-benchmark    100
allenai/wildjailbreak                         100
Name: count, dtype: int64



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>